In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp02/ex_2.py ──
"""
Shared infrastructure for MLFP02 Exercise 2 — Parameter Estimation and Inference.

Contains: Singapore economic data loading, log-likelihood objectives for
Normal / Student-t / Laplace, profile LR CI helpers, bootstrap utilities,
plot output directory and save helper.

Technique-specific narrative (which scenario, which interpretation) belongs
in the per-technique files in ``modules/mlfp02/solutions/ex_2/``.
"""

from pathlib import Path
from typing import Callable

import numpy as np
import plotly.graph_objects as go
import polars as pl
from scipy import stats
from scipy.optimize import minimize


# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp02_ex2_mle_theory"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig: go.Figure, filename: str) -> Path:
    """Write a Plotly figure to the exercise output directory."""
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore economic indicators
# ════════════════════════════════════════════════════════════════════════

SINGAPORE_ECON_DATASET = ("mlfp01", "economic_indicators.csv")


def load_singapore_econ() -> pl.DataFrame:
    """Load Singapore economic indicators (GDP, inflation, unemployment)."""
    loader = MLFPDataLoader()
    return loader.load(*SINGAPORE_ECON_DATASET)


def extract_series(df: pl.DataFrame, column: str) -> np.ndarray:
    """Drop nulls and return a float64 numpy array for a given column."""
    return df[column].drop_nulls().to_numpy().astype(np.float64)


def load_gdp_growth() -> np.ndarray:
    """Convenience: GDP growth (annualised quarterly %) as a 1-D numpy array."""
    return extract_series(load_singapore_econ(), "gdp_growth_pct")


# ════════════════════════════════════════════════════════════════════════
# LIKELIHOOD OBJECTIVES
# ════════════════════════════════════════════════════════════════════════
#
# All objectives are NEGATIVE log-likelihoods so they can be passed
# directly to scipy.optimize.minimize. Location/scale parameters are
# reparameterised where needed (log-sigma) to keep the optimiser in a
# valid region without explicit bounds.


def neg_log_likelihood_normal(params: np.ndarray, x: np.ndarray) -> float:
    """-log L for X ~ N(mu, sigma^2). params = [mu, log_sigma]."""
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    if sigma <= 0:
        return np.inf
    return float(-np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma)))


def neg_log_likelihood_t(params: np.ndarray, x: np.ndarray) -> float:
    """-log L for X ~ t(df, mu, scale). params = [df, mu, scale]."""
    df, mu, scale = params
    if df <= 0 or scale <= 0:
        return np.inf
    return float(-np.sum(stats.t.logpdf(x, df=df, loc=mu, scale=scale)))


def fit_normal_mle(x: np.ndarray) -> dict:
    """Fit Normal MLE numerically via L-BFGS-B. Returns mu, sigma, loglik, result."""
    x0 = np.array([float(x.mean()), float(np.log(x.std() + 1e-8))])
    result = minimize(
        neg_log_likelihood_normal,
        x0,
        args=(x,),
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return {
        "mu": float(result.x[0]),
        "sigma": float(np.exp(result.x[1])),
        "loglik": float(-result.fun),
        "converged": bool(result.success),
        "result": result,
    }


def fit_student_t_mle(x: np.ndarray) -> dict:
    """Fit Student-t MLE via Nelder-Mead. Returns df, mu, scale, loglik."""
    x0 = np.array([5.0, float(x.mean()), float(x.std() + 1e-8)])
    result = minimize(neg_log_likelihood_t, x0, args=(x,), method="Nelder-Mead")
    return {
        "df": float(result.x[0]),
        "mu": float(result.x[1]),
        "scale": float(result.x[2]),
        "loglik": float(-result.fun),
        "converged": bool(result.success),
        "result": result,
    }


# ════════════════════════════════════════════════════════════════════════
# FISHER INFORMATION + CONFIDENCE INTERVALS
# ════════════════════════════════════════════════════════════════════════
#
# For N(mu, sigma^2):
#   Var(mu_hat)   = sigma^2 / n         =>  SE(mu_hat)  = sigma / sqrt(n)
#   Var(sigma_hat) ~ sigma^2 / (2n)     =>  SE(sigma_hat) = sigma / sqrt(2n)
#
# These come directly from inverting the Fisher information matrix at
# the MLE.


def normal_fisher_standard_errors(sigma: float, n: int) -> tuple[float, float]:
    """Return (SE_mu, SE_sigma) from Fisher information at the Normal MLE."""
    se_mu = sigma / np.sqrt(n)
    se_sigma = sigma / np.sqrt(2 * n)
    return float(se_mu), float(se_sigma)


def wald_ci(estimate: float, se: float, alpha: float = 0.05) -> tuple[float, float]:
    """Symmetric Wald CI: estimate ± z_{1-alpha/2} * SE."""
    z = float(stats.norm.ppf(1 - alpha / 2))
    return (estimate - z * se, estimate + z * se)


def profile_lr_ci_normal_mu(
    x: np.ndarray,
    mu_hat: float,
    sigma_hat: float,
    loglik_at_mle: float,
    alpha: float = 0.05,
    grid_width_in_se: float = 4.0,
    n_grid: int = 500,
) -> tuple[tuple[float, float], np.ndarray, np.ndarray]:
    """Profile likelihood 1-alpha CI for the Normal mean.

    The CI is the set of mu where 2*(loglik_at_mle - loglik(mu)) < chi^2_{1-alpha, df=1}.

    Returns (ci, mu_grid, loglik_values) so the caller can plot the profile.
    """
    n = len(x)
    se_mu = sigma_hat / np.sqrt(n)
    threshold = float(stats.chi2.ppf(1 - alpha, df=1)) / 2.0

    mu_grid = np.linspace(
        mu_hat - grid_width_in_se * se_mu,
        mu_hat + grid_width_in_se * se_mu,
        n_grid,
    )
    loglik_values = np.array(
        [-neg_log_likelihood_normal([mu, np.log(sigma_hat)], x) for mu in mu_grid]
    )
    lr_values = loglik_at_mle - loglik_values
    mask = lr_values <= threshold
    if mask.any():
        ci = (float(mu_grid[mask][0]), float(mu_grid[mask][-1]))
    else:
        # Fallback: Wald CI
        ci = wald_ci(mu_hat, se_mu, alpha)
    return ci, mu_grid, loglik_values


# ════════════════════════════════════════════════════════════════════════
# MAP — NORMAL LIKELIHOOD + NORMAL PRIOR ON THE MEAN
# ════════════════════════════════════════════════════════════════════════


def make_map_objective(
    prior_mean: float, prior_std: float
) -> Callable[[np.ndarray, np.ndarray], float]:
    """Return a neg_map_objective(params, x) closure for the given Normal prior on mu."""

    def neg_map_objective(params: np.ndarray, x: np.ndarray) -> float:
        mu, log_sigma = params
        sigma = np.exp(log_sigma)
        if sigma <= 0:
            return np.inf
        nll = -np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma))
        neg_log_prior = -stats.norm.logpdf(mu, loc=prior_mean, scale=prior_std)
        return float(nll + neg_log_prior)

    return neg_map_objective


def fit_normal_map(x: np.ndarray, prior_mean: float, prior_std: float) -> dict:
    """Fit MAP for Normal likelihood with a Normal(prior_mean, prior_std^2) prior on mu."""
    neg_map = make_map_objective(prior_mean, prior_std)
    x0 = np.array([float(x.mean()), float(np.log(x.std() + 1e-8))])
    result = minimize(
        neg_map,
        x0,
        args=(x,),
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return {
        "mu": float(result.x[0]),
        "sigma": float(np.exp(result.x[1])),
        "converged": bool(result.success),
        "result": result,
    }


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP UTILITIES
# ════════════════════════════════════════════════════════════════════════


def bootstrap_statistic(
    x: np.ndarray,
    statistic: Callable[[np.ndarray], float],
    n_boot: int = 10_000,
    seed: int | None = 42,
) -> np.ndarray:
    """Nonparametric bootstrap: resample x with replacement, apply statistic."""
    rng = np.random.default_rng(seed)
    n = len(x)
    out = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        out[i] = statistic(rng.choice(x, size=n, replace=True))
    return out


def bootstrap_percentile_ci(
    boot_samples: np.ndarray, alpha: float = 0.05
) -> tuple[float, float]:
    lo = float(np.percentile(boot_samples, 100 * alpha / 2))
    hi = float(np.percentile(boot_samples, 100 * (1 - alpha / 2)))
    return lo, hi


# ════════════════════════════════════════════════════════════════════════
# AIC / BIC
# ════════════════════════════════════════════════════════════════════════


def aic(k: int, loglik: float) -> float:
    return 2 * k - 2 * loglik


def bic(k: int, loglik: float, n: int) -> float:
    return k * float(np.log(n)) - 2 * loglik


# ════════════════════════════════════════════════════════════════════════
# DEFAULTS — SAMPLE SIZES, SEEDS, PRIOR VALUES
# ════════════════════════════════════════════════════════════════════════
#
# These constants are referenced by multiple technique files. Change them
# once here and every file picks up the update.

DEFAULT_SEED: int = 42
DEFAULT_N_BOOT: int = 10_000
DEFAULT_N_CLT_REPS: int = 5000

# Singapore prior on quarterly GDP growth — long-run open-economy estimate
# that we use to illustrate MAP shrinkage.
GDP_PRIOR_MEAN: float = 3.5  # %
GDP_PRIOR_STD: float = 1.5  # %




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 2.4: MLE Failure Modes
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Diagnose MLE failure case 1: small n — biased, high-variance sigma
#   - Diagnose MLE failure case 2: multimodal data — mean between modes
#   - Diagnose MLE failure case 3: misspecified likelihood — tail risk
#   - Know when MLE is trustworthy and when alternatives are needed
#   - Connect each failure to its remedy (Bessel, GMM, model selection)
#
# PREREQUISITES: 03_map_estimation.py (MAP as a remedy for small n)
# ESTIMATED TIME: ~35 minutes
#
# TASKS (5-phase R10):
#   1. Theory — three ways MLE can mislead
#   2. Build — simulations for each failure mode
#   3. Train — quantify bias, bimodality, and tail underestimation
#   4. Visualise — bimodal histogram + tail comparison table
#   5. Apply — MAS stress testing: why Normal underestimates crises
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from kailash_ml import ModelVisualizer
from scipy import stats



## THEORY — Three MLE Failure Modes


In [ ]:
#
# FAILURE 1 — SMALL n: MLE sigma biased low by (n-1)/n. CIs too narrow.
# FAILURE 2 — MULTIMODAL: single Normal puts mean in the "valley".
# FAILURE 3 — MISSPECIFIED: Normal underestimates heavy tails.



## TASK 2 — BUILD: Small n Simulation


In [ ]:
econ = load_singapore_econ()
gdp_growth = extract_series(econ, "gdp_growth_pct")
n_gdp = len(gdp_growth)

rng = np.random.default_rng(seed=DEFAULT_SEED)

print(f"\n=== MLE Failure Case 1: Small n ===")

small_ns = [3, 5, 10]
n_trials_per = 2000

for n_s in small_ns:
    mle_mus = []
    mle_sigmas = []
    for _ in range(n_trials_per):
        # TODO: Draw a random subsample of size n_s from gdp_growth
        # (without replacement), then compute the sample mean and
        # sample std (ddof=0).
        # Hint: rng.choice(gdp_growth, size=n_s, replace=False)
        sample = ____
        mle_mus.append(____)
        mle_sigmas.append(____)

    mu_bias = np.mean(mle_mus) - gdp_growth.mean()
    sigma_bias = np.mean(mle_sigmas) - gdp_growth.std(ddof=0)
    mu_var = np.var(mle_mus)
    sigma_var = np.var(mle_sigmas)

    print(f"\nn={n_s}: (over {n_trials_per} random subsets)")
    print(f"  mu_hat: bias={mu_bias:+.4f}%, variance={mu_var:.4f}")
    print(f"  sigma_hat: bias={sigma_bias:+.4f}%, variance={sigma_var:.4f}")
    print(f"  sigma_hat negative bias confirms MLE underestimates sigma for small n")



### Checkpoint 1


In [ ]:
mle_sigmas_3 = [
    rng.choice(gdp_growth, size=3, replace=False).std(ddof=0) for _ in range(1000)
]
assert np.mean(mle_sigmas_3) < gdp_growth.std(
    ddof=0
), "MLE sigma_hat should be biased low for n=3"
print("\n--- Checkpoint 1 passed --- small-n bias demonstrated\n")



## TASK 3a — TRAIN: Multimodal Data Failure


In [ ]:
print(f"\n=== MLE Failure Case 2: Multimodal Data ===")

# Simulate two economic regimes: pre-COVID vs COVID
pre_covid = rng.normal(loc=4.0, scale=1.2, size=30)
covid_shock = rng.normal(loc=-5.0, scale=3.0, size=10)
bimodal_data = np.concatenate([pre_covid, covid_shock])

# TODO: Compute the MLE mean and sigma of the bimodal data.
# Hint: bimodal_data.mean() and bimodal_data.std(ddof=0)
bimodal_mle_mu = ____
bimodal_mle_sigma = ____

# Bimodality coefficient
bimodality_coeff = (stats.skew(bimodal_data) ** 2 + 1) / (
    stats.kurtosis(bimodal_data, fisher=True)
    + 3
    * (
        (len(bimodal_data) - 1) ** 2
        / ((len(bimodal_data) - 2) * (len(bimodal_data) - 3))
    )
)

print(f"Bimodal data: {len(bimodal_data)} observations (two economic regimes)")
print(f"Mode 1 (pre-COVID): mean=4.0%, n=30")
print(f"Mode 2 (COVID shock): mean=-5.0%, n=10")
print(f"\nSingle Normal MLE: mu={bimodal_mle_mu:.2f}%, sigma={bimodal_mle_sigma:.2f}%")
print(f"Bimodality coefficient: {bimodality_coeff:.3f} (>0.555 -> bimodal)")
print(f"Problem: MLE estimate {bimodal_mle_mu:.2f}% lies BETWEEN the two modes!")
print(f"  -> No actual observation is near the MLE mean")
print(f"  -> The model assigns low probability to BOTH clusters of real data")
print(f"\nRemedy: Gaussian Mixture Model (GMM) or regime-switching model")



### Checkpoint 2


In [ ]:
assert (
    bimodal_mle_mu > -5.0 and bimodal_mle_mu < 4.0
), "MLE mean should fall between the two modes"
print("\n--- Checkpoint 2 passed --- multimodal failure demonstrated\n")



## TASK 3b — TRAIN: Misspecified Likelihood


In [ ]:
print(f"\n=== MLE Failure Case 3: Misspecified Likelihood ===")

rng_t = np.random.default_rng(seed=77)
shock_data = rng_t.standard_t(df=3, size=100) * 2.0 + 2.5

# Fit Normal MLE
normal_mle_mu = shock_data.mean()
normal_mle_sigma = shock_data.std(ddof=0)

# TODO: Fit a Student-t distribution to the shock data using
# fit_student_t_mle(). It returns a dict with keys: df, mu, scale.
# Hint: fit_student_t_mle(shock_data)
t_result = ____
t_df = t_result["df"]
t_mu = t_result["mu"]
t_scale = t_result["scale"]

# Compare tail risk at different percentiles
print(
    f"{'Percentile':<12} {'Normal':>10} {'t-dist':>10} {'Empirical':>10} {'Normal Error':>12}"
)
print("-" * 60)
for pctile in [90, 95, 99, 99.5]:
    normal_q = stats.norm.ppf(pctile / 100, loc=normal_mle_mu, scale=normal_mle_sigma)
    t_q = stats.t.ppf(pctile / 100, df=t_df, loc=t_mu, scale=t_scale)
    emp_q = np.percentile(shock_data, pctile)
    error = abs(normal_q - emp_q)
    print(
        f"{pctile:>10}th  {normal_q:>10.2f} {t_q:>10.2f} {emp_q:>10.2f} {error:>12.2f}"
    )

print(f"\nt-distribution df = {t_df:.1f} (lower -> heavier tails)")
print(f"Normal systematically underestimates extreme events")



### Checkpoint 3


In [ ]:
normal_99 = stats.norm.ppf(0.99, loc=normal_mle_mu, scale=normal_mle_sigma)
t_99 = stats.t.ppf(0.99, df=t_df, loc=t_mu, scale=t_scale)
assert t_99 > normal_99, "t-distribution 99th percentile should exceed Normal's"
print("\n--- Checkpoint 3 passed --- misspecification and tail risk demonstrated\n")



## TASK 4 — VISUALISE: Bimodal Failure


In [ ]:
viz = ModelVisualizer()

fig = viz.histogram(
    bimodal_data,
    title="Bimodal GDP Growth (Pre-COVID + COVID Shock)",
    x_label="GDP Growth (%)",
)
fig.add_vline(x=bimodal_mle_mu, line_dash="dash", annotation_text="MLE mean")
save_figure(fig, "ex2_04_bimodal_failure.html")
print("Saved: ex2_04_bimodal_failure.html")



### Checkpoint 4


In [ ]:
print("\n--- Checkpoint 4 passed --- bimodal failure visualised\n")



## TASK 5 — APPLY: MAS Stress Testing — Why Normal Underestimates Crises


In [ ]:
print(f"\n=== APPLY: MAS Stress Testing ===")

threshold = -5.0
# TODO: Compute the probability of GDP growth < threshold under
# Normal and t-distribution models.
# Hint: stats.norm.cdf(threshold, loc=..., scale=...)
#       stats.t.cdf(threshold, df=..., loc=..., scale=...)
prob_normal = ____
prob_t = ____
empirical_fraction = np.mean(shock_data < threshold)

print(f"Stress threshold: GDP growth < {threshold}%")
print(f"Normal model probability:  {prob_normal:.6f} ({prob_normal*100:.4f}%)")
print(f"t-distribution probability: {prob_t:.6f} ({prob_t*100:.4f}%)")
print(
    f"Empirical frequency:       {empirical_fraction:.4f} ({empirical_fraction*100:.2f}%)"
)
print(
    f"\nNormal UNDERESTIMATES crisis probability by {prob_t/max(prob_normal, 1e-12):.1f}x"
)
print(
    f"If MAS sets capital requirements using the Normal model, banks"
    f"\nwill hold insufficient reserves for tail events."
)



### Checkpoint 5


In [ ]:
assert prob_t > prob_normal, "t-dist assigns more probability to tail events"
print("\n--- Checkpoint 5 passed --- MAS stress testing application complete\n")



## REFLECTION


- MLE failure 1: small n -> biased sigma (Bessel/MAP remedies)
  - MLE failure 2: multimodal -> mean between modes (GMM remedy)
  - MLE failure 3: misspecification -> wrong tails (model selection)
  - Always visualise before fitting — histograms reveal bimodality
  - Heavy-tailed distributions (Student-t) for financial risk
  - Real-world impact: stress testing with correct tail probabilities

  NEXT: In 05_model_selection.py, you'll use AIC/BIC to formally
  compare distribution families and bootstrap for non-standard
  statistics (median, trimmed mean, IQR).


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print("--- Exercise 2.4 complete --- MLE Failure Modes")

